In [1]:
import pandas as pd

calendar = pd.read_csv("calendar_rates.csv")
listings = pd.read_csv("listings.csv")

In [2]:
calendar['listing_id'] = calendar['listing_id'].astype(str)
listings['listing_id'] = listings['listing_id'].astype(str)

In [3]:
print(listings['superhost'].unique())

[False  True]


In [4]:
listings['superhost_flag'] = listings['superhost'].map({'True': 1, 'False': 0})

In [5]:
listings['superhost_flag'] = listings['superhost'].astype(int)

In [6]:
listings = listings[['listing_id', 'superhost_flag', 'bedrooms', 'beds', 'baths']]

In [7]:
df = calendar.merge(listings, on='listing_id', how='inner')

In [8]:
print(df.shape)
print(df['superhost_flag'].value_counts())

(6929, 21)
superhost_flag
1    4769
0    2160
Name: count, dtype: int64


In [9]:
df = df.dropna(subset=['occupancy', 'superhost_flag'])

In [10]:
import statsmodels.formula.api as smf

model = smf.ols(
    formula="occupancy ~ superhost_flag + rate_avg + bedrooms + beds + baths",
    data=df
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              occupancy   R-squared:                       0.026
Model:                            OLS   Adj. R-squared:                  0.025
Method:                 Least Squares   F-statistic:                     29.45
Date:                Fri, 27 Mar 2026   Prob (F-statistic):           1.31e-29
Time:                        01:44:09   Log-Likelihood:                -1651.9
No. Observations:                5471   AIC:                             3316.
Df Residuals:                    5465   BIC:                             3355.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          0.2443      0.014     18.

In [11]:
model_base = smf.ols(
    formula="occupancy ~ rate_avg + bedrooms + beds + baths",
    data=df
).fit()

print("\n=== BASELINE MODEL (NO SUPERHOST) ===")
print(model_base.summary())


=== BASELINE MODEL (NO SUPERHOST) ===
                            OLS Regression Results                            
Dep. Variable:              occupancy   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                  0.010
Method:                 Least Squares   F-statistic:                     14.31
Date:                Fri, 27 Mar 2026   Prob (F-statistic):           1.26e-11
Time:                        01:44:09   Log-Likelihood:                -1696.1
No. Observations:                5471   AIC:                             3402.
Df Residuals:                    5466   BIC:                             3435.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    

In [12]:
print("\nOccupancy by demand period:")
print(df.groupby('demand_period')['occupancy'].mean())

print("\nRevenue by demand period:")
print(df.groupby('demand_period')['revenue'].mean())


Occupancy by demand period:
demand_period
off-peak    0.250762
peak        0.245862
shoulder    0.252764
Name: occupancy, dtype: float64

Revenue by demand period:
demand_period
off-peak    1206.596271
peak        1314.155671
shoulder    1400.588976
Name: revenue, dtype: float64


In [13]:
model2 = smf.ols(
    formula="occupancy ~ superhost_flag * C(demand_period) + rate_avg",
    data=df
).fit()

print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:              occupancy   R-squared:                       0.013
Model:                            OLS   Adj. R-squared:                  0.012
Method:                 Least Squares   F-statistic:                     14.96
Date:                Fri, 27 Mar 2026   Prob (F-statistic):           4.37e-17
Time:                        01:44:09   Log-Likelihood:                -2059.5
No. Observations:                6929   AIC:                             4133.
Df Residuals:                    6922   BIC:                             4181.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [16]:
coef = model.params['superhost_flag']
pval = model.pvalues['superhost_flag']

print(f"Coefficient: {coef}")
print(f"P-value: {pval}")

Coefficient: 0.09310494973277504
P-value: 5.474945098806601e-21
